# General pipeline

#### General pre-processing, cleaning and model training steps relevant for all data versions.
#### It's aim is to experiment with different combinations of features to determine predictive value of features added for enrichment.
#### The pipeline is based on TM-RugPull initial analysis and includes only steps that are relevant and were proven to be useful for the dataset.

In [1]:
 # Loading data from .xlsx file

import pandas as pd
import numpy as np
import matplotlib.pyplot as pyplot

file = 'data/TM-RugPull_prepared_for_enrichment.xlsx'

data = pd.read_excel(file)

# Remove a space in the end of some column names
data.columns = data.columns.str.strip()

print(data.shape)

pd.set_option('display.max_columns', None)

data.head(5)

(971, 27)


,Project Title,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Sign,Blockchain Type,Smart Contract (online),smart Contract (offline),website,x profile,class,project starting date,project end date,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2),Contract address
0,HyperVerse Token (HVT),7.650000e+00,1.500000e-01,9.100000e-06,8.000000e-07,BSC,623909,25752,0.009537,4.929203e-02,166600000000100,HVT,POSA,sourse code,CODE,https://thehyperverse.net/index.html,https://twitter.com/HyperVerse6,scam,2022-01-27,2023-07-14,148,148,29,8,74,47,0xaafa10755b3b1dbf46e86d973c3f27f3671ed9db
1,Fintoch,1.795000e-11,1.907000e-11,1.727000e-10,1.769000e-10,BSC,"1,492,842","147,791",2487.228167,4.503032e+07,34403.74594,BEP-20 TOKEN*,POSA,sourse code,CODE,https://web.archive.org/web/20230603123631/htt...,NaN,scam,2022-07-12,2023-06-19,820,167,4530,1590,48,4,0x19a00e359990ec7daf6e9dd9a2fb7664014bb5f7
2,Flare Token,1.955000e-03,5.519000e-04,4.158000e-04,2.838000e-04,BSC,"184,694",15173,890171363872314.625,6.695544e+17,10000000000,Flare,POSA,sourse code,CODE,https://pipeflare.io/,https://x.com/MetaFlareToken,scam,2021-10-24,2022-11-24,421,156,6,5,1710,1150,0x85aa3f04e539e426cbb55c0d584ea99cfe1d96a1
3,Safuu Protocol,2.070000e+02,2.110000e+02,7.000000e+01,2.400000e+01,BSC,"275,530",151979,24573310762.585602,8.139816e+14,61634066.59803,SAFUU,POSA,sourse code,CODE,https://safuu.com/,https://x.com/safuuxofficial,scam,2022-02-03,2022-08-13,327,323,0,0,5,4,0xe5ba47fd94cb645ba4119222e34fb33f59c7cd90
4,SCT,2.986000e-01,1.659000e-01,1.831000e-01,1.552000e-01,BSC,8445,"6,126\n",4252172.090125,1.410127e+05,"42,896,736.739367",SCT,POSA,sourse code,CODE,https://supercells.jp/en/,https://x.com/scttoken,scam,2023-02-27,2024-07-09,4990000,345000,6,3,1830,594,0x4ee98216499b81a9942e7aa77970b68c792ff679


In [2]:
# Create a test set and a validation set from the raw data to avoid data leakage
# A split is temporal, preserving project period balance (based on start date), so that both old and new projects are presented in each split
# Validation set size is about 10,5% and test set size is about 24,5% from the whole data set

# Reference: https://stackoverflow.com/questions/45516424/sklearn-train-test-split-on-pandas-stratify-by-multiple-columns

from sklearn.model_selection import train_test_split

# Group years into broader periods
def assign_project_period(year):
    if year < 2021:
        return 'before_2021'
    elif year in [2021, 2022, 2023]:
        return str(year)
    elif year in [2024, 2025]:
        return '2024_2025'
    else:
        return 'unknown'


# Create test and validation sets using temporal stratified split
def create_temporal_stratified_split():
    # Convert project starting date to datetime and extract year
    data['project starting date'] = pd.to_datetime(data['project starting date'], errors='coerce')
    data['project starting year'] = data['project starting date'].dt.year
    data['project period'] = data['project starting year'].apply(assign_project_period)
    # Define seed
    seed = 7
    # Combined stratification label for the full dataset (class and project period)
    combined_labels = data['class'].astype(str) + '|' + data['project period'].astype(str)
    # Split the data first on train set and set for test and validation
    train_set, test_and_val_set = train_test_split(data, test_size=0.35, random_state=seed, stratify=combined_labels)
    # Combined stratification label for test_and_val_set (class and project period)
    subset_combined_labels = (test_and_val_set['class'].astype(str) + '|' + test_and_val_set['project period'].astype(str))
    #Split the part for test and validation into test set and validation set
    test_set, val_set = train_test_split(test_and_val_set, test_size=0.3, random_state=seed, stratify=subset_combined_labels)

    return train_set, test_set, val_set

train_set, test_set, val_set = create_temporal_stratified_split()

# Output the shapes to verify the splits
print("Training set shape:", train_set.shape)
print("Test set shape:", test_set.shape)
print("Validation set shape:", val_set.shape)

# Create a list of sets to perform further feature engineering on all subsets of data
data_sets = [train_set, test_set, val_set]

Training set shape: (631, 29)
Test set shape: (238, 29)
Validation set shape: (102, 29)


In [3]:
# Verify that there is no overlap between sets, no data leak at this stage
print("Overlap between train and test:", np.intersect1d(train_set.index, test_set.index).size)
print("Overlap between train and validation:", np.intersect1d(train_set.index, val_set.index).size)
print("Overlap between test and validation:", np.intersect1d(test_set.index, val_set.index).size)

Overlap between train and test: 0
Overlap between train and validation: 0
Overlap between test and validation: 0


In [4]:
#TODO: compute project duration


In [5]:
# Delete columns that are not useful for further analysis

for data_set in data_sets:
    data_set.drop(columns=['Project Title', 'Sign', 'website', 'x profile', 'Smart Contract (online)', 'smart Contract (offline)', 'project starting date', 'project end date', 'project starting year', 'project period', 'Contract address'], inplace=True)

# Check the result on train set
train_set.head(10)

,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Blockchain Type,class,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
721,2.727200e+03,1.249400e+04,6.396500e+04,1.034420e+05,BSC,80859641,2175554,62887128.485925,2.266745e+09,604999.99995,POSA,normal,669,10,1890,1130,532,2
921,3.400000e+01,1.240000e+01,5.780000e+00,2.000000e+00,ETH,130689,6727,107260696815604.09375,2.309422e+17,1000000000,POS,normal,1890,994,42,6,745,437
366,2.210860e-01,1.384800e-01,2.311000e-02,7.309000e-03,ETH,31,29,34401173646938632,9.988571e+08,1000000000,POS,scam,2,1,1300,1280,16,12
585,7.137000e+01,1.989000e+01,1.690000e+01,2.852000e+01,ETH,1005900,65748,43528554786.590347,3.300932e+14,100000000,POS,normal,159,64,7,5,9060,2400
221,2.216200e-08,5.979000e-09,3.826000e-09,5.146000e-09,ETH,824,263,2313261542507210923371796745700743719393367410...,6.224664e+07,5000000000,POS,scam,984,10,0,0,1,1
406,4.900000e-02,2.130000e-02,6.500000e-03,7.660000e-03,ETH,9972,1469,166.157816,2.303429e+03,7040,POS,scam,96,85,1670,1040,617,602
511,6.900000e-03,2.500000e-02,1.080000e-02,7.100000e-03,ETH,111,73,778488345570885162112122880,1.908837e+27,1000000000000000,POS,scam,38,9,4,4,22,13
798,1.750000e+00,2.900000e-01,2.240000e-01,9.500000e-02,ETH,"68,441",2747,9089622705049.445312,2.107311e+15,320000000,POS,normal,46,10,2,1,89,6
517,1.170000e-02,4.100000e-03,6.100000e-03,6.000000e-04,ETH,27,24,14505556378312100,1.641595e+16,1000000000,POS,scam,1,0,61,8,2,0
470,1.280000e-08,1.310000e-08,1.480000e-07,1.530000e-07,ETH,287,172,14048160827256879448064,0.000000e+00,69420000000000,POS,scam,2,2,0,0,0,0


In [6]:
# Encode class label

from sklearn.preprocessing import LabelEncoder

for data_set in data_sets:
    le = LabelEncoder()
    data_set['class'] = data_set['class'].map({'normal': 0, 'scam': 1})

train_set

,MaxPrice (Quarter 1),MaxPrice (Quarter 2),MaxPrice (Quarter 3),MaxPrice (Quarter 4),Blockchain,the number of Transactions,Token concentration ratio per holder,Total Variance,Variance of holders with more than 1% tokens,Token balance,Blockchain Type,class,Google results for project title (first day),Google results for project title (project duration/2),Google results for project website (first day),Google results for project website (duration/2),Google results for project x profile (first days),Google results for project x profile (duration/2)
721,2.727200e+03,1.249400e+04,6.396500e+04,1.034420e+05,BSC,80859641,2175554,62887128.485925,2.266745e+09,604999.99995,POSA,0,669,10,1890,1130,532,2
921,3.400000e+01,1.240000e+01,5.780000e+00,2.000000e+00,ETH,130689,6727,107260696815604.09375,2.309422e+17,1000000000,POS,0,1890,994,42,6,745,437
366,2.210860e-01,1.384800e-01,2.311000e-02,7.309000e-03,ETH,31,29,34401173646938632,9.988571e+08,1000000000,POS,1,2,1,1300,1280,16,12
585,7.137000e+01,1.989000e+01,1.690000e+01,2.852000e+01,ETH,1005900,65748,43528554786.590347,3.300932e+14,100000000,POS,0,159,64,7,5,9060,2400
221,2.216200e-08,5.979000e-09,3.826000e-09,5.146000e-09,ETH,824,263,2313261542507210923371796745700743719393367410...,6.224664e+07,5000000000,POS,1,984,10,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,4.443000e-03,7.346000e-03,6.244550e-03,7.900000e-03,ETH,22554,1817,190157796754184.1875,6.215754e+16,1000000000,POS,1,156,10,1,1,11,55
656,2.400000e-02,1.070000e-03,6.400000e-04,3.600000e-04,ETH,25,2729,27408742440935.449219,4.683034e+15,1000000000,POS,0,3,2,8,8,668,254
966,3.400000e+00,1.800000e+00,2.150000e+00,1.100000e+01,BSC,15763,5632,32177196871579.179688,9.233344e+15,1000000000,POSA,0,15200,4000,318000,80500,84,67
123,4.029000e-03,2.285000e-03,1.419360e-03,8.998000e-04,BSC,540,288,5423455969341.307617,9.276477e+13,100000000,POSA,1,0,0,7,6,0,0


In [7]:
# Transfer all data to numeric values

# Columns with object datatype
cols_to_clean = [
    'the number of Transactions',
    'Token concentration ratio per holder',
    'Token balance',
    'first deposits',
    'Google results for project website (first day)',
    'Google results for project website (duration/2)'
]

# Apply to all data splits
data_sets = [train_set, test_set, val_set]

for i, data_set in enumerate(data_sets):
    for col in cols_to_clean:
        data_sets[i][col] = pd.to_numeric(
            data_set[col].astype(str)
                   .str.replace('\xa0', '', regex=False)        # Remove non-breaking whitespaces
                   .str.replace(',', '', regex=False)           # Strip comas separating numeric values
                   .str.strip(),                                # Remove surrounding whitespaces
            errors='coerce'                                     # If cannot parse, put NaN
        )

train_set, test_set, val_set = data_sets

# Verify the result
print("Datatypes:\n", train_set[cols_to_clean].dtypes)
print("\nMissing values:\n", train_set[cols_to_clean].isnull().sum())
print("\nColumns description\n", train_set[cols_to_clean].describe())

KeyError: 'first deposits'